In [ ]:
import hashlib
import time
import json
import threading
import random

# Shared ledger (JSON-based structure)
ledger = {
    "clients": {},  # Stores client identity hashes
    "transactions": []  # Stores transactions
}
ledger_lock = threading.Lock()

# Proof of Work function
def proof_of_work(data, difficulty=2):
    nonce = 0
    prefix = '0' * (difficulty * 2)
    start_time = time.time()

    while True:
        text = f"{data}{nonce}".encode()
        hash_value = hashlib.sha256(text).hexdigest()

        if hash_value.startswith(prefix):
            end_time = time.time()
            return nonce, hash_value, round(end_time - start_time, 4)

        nonce += 1

# Server class
def server_thread(server_id, difficulty=2):
    global ledger

    client_id = f"client_{server_id}"  # Simulated client
    client_hash = hashlib.sha256(client_id.encode()).hexdigest()
    data = f"Transaction_{server_id}"  # Data for PoW

    print(f"Server {server_id} starting Proof of Work...")
    nonce, final_hash, time_taken = proof_of_work(data, difficulty)
    print(f"Server {server_id} solved PoW: Nonce={nonce}, Hash={final_hash}, Time={time_taken}s")

    with ledger_lock:
        ledger["clients"][client_id] = client_hash
        ledger["transactions"].append({
            "server": server_id,
            "nonce": nonce,
            "hash": final_hash,
            "time": time_taken
        })

    # Only one server is designated as the validator
    if server_id == 1:  # Arbitrarily choosing server 1 as the validator
        validate_transactions()

# Validator function
def validate_transactions():
    with ledger_lock:
        print("Validating transactions...")
        for tx in ledger["transactions"]:
            data = f"Transaction_{tx['server']}"
            nonce, hash_value, _ = proof_of_work(data, difficulty=2)
            if hash_value == tx["hash"]:
                print(f"✅ Transaction from Server {tx['server']} is VALID!")
            else:
                print(f"❌ Transaction from Server {tx['server']} is INVALID!")

# Initialize server threads
threads = []
num_servers = 3
for i in range(1, num_servers + 1):
    t = threading.Thread(target=server_thread, args=(i,))
    threads.append(t)
    t.start()

# Wait for all threads to complete
for t in threads:
    t.join()

# Print final ledger
with ledger_lock:
    print("Final Ledger State:")
    print(json.dumps(ledger, indent=4))


Server 1 starting Proof of Work...
Server 2 starting Proof of Work...
Server 2 solved PoW: Nonce=5335, Hash=00002e4123dcbced59860850369471cfdde996491c7111545d415622d2658ba1, Time=0.0128sServer 3 starting Proof of Work...

Server 3 solved PoW: Nonce=9108, Hash=0000c52ccb53fe8bd53d31d2256e6540ba3afd4f48b64ae4f1d964b9efc9148d, Time=0.0515s
Server 1 solved PoW: Nonce=106801, Hash=00002070c3925f931ce37bd4d745bc908667afbf16aa1d6bf557df28132f5d4a, Time=0.2712s
Validating transactions...
✅ Transaction from Server 2 is VALID!
✅ Transaction from Server 3 is VALID!
✅ Transaction from Server 1 is VALID!
Final Ledger State:
{
    "clients": {
        "client_2": "2aba89f76f474748129329d2b985e7557271562e1d32e2a0281f6c343e751c1d",
        "client_3": "7d8aee718bd306916a163493ee608483c9a3447816c48b268d6b00c03eb24554",
        "client_1": "914e1681af0f4df7627b30092c5c2077d304fc0b0d05b3dab50e52d2a2d4f4a7"
    },
    "transactions": [
        {
            "server": 2,
            "nonce": 5335,
        

In [ ]:
import hashlib
import time
import json
import threading
import random

# Shared ledger (JSON-based structure)
ledger = {
    "clients": {},  # Stores client identity hashes
    "transactions": []  # Stores transactions
}
ledger_lock = threading.Lock()

# Proof of Work function
def proof_of_work(data, difficulty=2):
    nonce = 0
    prefix = '0' * (difficulty * 2)
    start_time = time.time()

    while True:
        text = f"{data}{nonce}".encode()
        hash_value = hashlib.sha256(text).hexdigest()

        if hash_value.startswith(prefix):
            end_time = time.time()
            return nonce, hash_value, round(end_time - start_time, 4)

        nonce += 1

# Server class
def server_thread(server_id, difficulty=2):
    global ledger

    client_id = f"client_{server_id}"  # Simulated client
    client_hash = hashlib.sha256(client_id.encode()).hexdigest()
    data = f"Transaction_{server_id}"  # Data for PoW

    print(f"Server {server_id} starting Proof of Work...")
    nonce, final_hash, time_taken = proof_of_work(data, difficulty)
    print(f"Server {server_id} solved PoW: Nonce={nonce}, Hash={final_hash}, Time={time_taken}s")

    with ledger_lock:
        ledger["clients"][client_id] = client_hash
        ledger["transactions"].append({
            "server": server_id,
            "nonce": nonce,
            "hash": final_hash,
            "time": time_taken
        })

    # Randomly selecting other servers to validate the winning transaction
    winning_server = server_id
    validator_servers = [s for s in range(1, 4) if s != winning_server]
    random.shuffle(validator_servers)

    for validator in validator_servers:
        threading.Thread(target=validate_transactions, args=(winning_server, validator)).start()

# Validator function
def validate_transactions(target_server, validator_server):
    with ledger_lock:
        print(f"Server {validator_server} validating transaction from Server {target_server}...")
        for tx in ledger["transactions"]:
            if tx['server'] == target_server:
                data = f"Transaction_{tx['server']}"
                nonce, hash_value, _ = proof_of_work(data, difficulty=2)
                if hash_value == tx["hash"]:
                    print(f"✅ Server {validator_server} validated transaction from Server {tx['server']} as VALID!")
                else:
                    print(f"❌ Server {validator_server} found transaction from Server {tx['server']} INVALID!")

# Initialize server threads
threads = []
num_servers = 3
for i in range(1, num_servers + 1):
    t = threading.Thread(target=server_thread, args=(i,))
    threads.append(t)
    t.start()

# Wait for all threads to complete
for t in threads:
    t.join()

# Print final ledger
with ledger_lock:
    print("Final Ledger State:")
    print(json.dumps(ledger, indent=4))

Server 1 starting Proof of Work...Server 2 starting Proof of Work...

Server 2 solved PoW: Nonce=5335, Hash=00002e4123dcbced59860850369471cfdde996491c7111545d415622d2658ba1, Time=0.0118sServer 3 starting Proof of Work...

Server 3 solved PoW: Nonce=9108, Hash=0000c52ccb53fe8bd53d31d2256e6540ba3afd4f48b64ae4f1d964b9efc9148d, Time=0.0174s
Server 1 validating transaction from Server 2...
✅ Server 1 validated transaction from Server 2 as VALID!
Server 3 validating transaction from Server 2...
✅ Server 3 validated transaction from Server 2 as VALID!
Server 2 validating transaction from Server 3...
Server 1 solved PoW: Nonce=106801, Hash=00002070c3925f931ce37bd4d745bc908667afbf16aa1d6bf557df28132f5d4a, Time=0.1678s
✅ Server 2 validated transaction from Server 3 as VALID!
Server 1 validating transaction from Server 3...
✅ Server 1 validated transaction from Server 3 as VALID!
Server 3 validating transaction from Server 1...
✅ Server 3 validated transaction from Server 1 as VALID!
Server 2 val